# Coletor de Letras
Busca dados do artista no Spotify + letras na API do Genius.

In [1]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, module='spotipy')

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd
import json
import lyricsgenius as genius
import os
import time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

True

### Importações

In [2]:
GENIUS_TOKEN = os.getenv("GENIUS_TOKEN")
SPOTIFY_TOKEN = os.getenv("SPOTIFY_TOKEN")
SPOTIFY_SECRET = os.getenv("SPOTIFY_SECRET")

OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

ARTIST_NAME = "Kendrick Lamar"

## Conectando ao Spotify

### Configuração

In [3]:
ccm = SpotifyClientCredentials(client_id=SPOTIFY_TOKEN, client_secret=SPOTIFY_SECRET)
sp = spotipy.Spotify(client_credentials_manager=ccm)

## Pegando infos do artista do Spotify

In [4]:
results = sp.search(f"artist:{ARTIST_NAME}")
artist_id = results['tracks']['items'][0]['artists'][0]['uri']
print(f"Artist: {ARTIST_NAME}")
print(f"URI: {artist_id}")

Artist: Kendrick Lamar
URI: spotify:artist:2YZyLoL8N0Wb9xBt1NhZWg


In [5]:
albums = sp.artist_albums(artist_id, country="US", limit=50)
print(albums['next'])

https://api.spotify.com/v1/artists/2YZyLoL8N0Wb9xBt1NhZWg/albums?offset=50&limit=50&include_groups=album,single,compilation,appears_on


### Buscando Álbuns do Artista

In [6]:
album_ids = [album['uri'] for album in albums['items']]
print(len(album_ids))
print(album_ids[0])

50
spotify:album:0hvT3yIEysuuvkK73vgdcW


In [7]:
all_tracks = []
for album_id in album_ids:
    tracks = sp.album_tracks(album_id, limit=50)
    all_tracks.append(tracks)

In [8]:
for tracks, album in zip(all_tracks, albums.get('items')):
    print(
        len(tracks.get('items')),
        "\t",
        album.get('name')
    )

12 	 GNX
19 	 Mr. Morale & The Big Steppers
14 	 Black Panther The Album Music From And Inspired By
14 	 DAMN. COLLECTORS EDITION.
14 	 DAMN.
8 	 untitled unmastered.
16 	 To Pimp A Butterfly
17 	 good kid, m.A.A.d city (Deluxe)
14 	 good kid, m.A.A.d city
15 	 Section.80
12 	 Overly Dedicated
1 	 Super Bowl LIX Halftime Show (Live)
1 	 Not Like Us
1 	 meet the grahams
1 	 euphoria
1 	 The Hillbillies
1 	 The Heart Part 5
1 	 family ties (with Kendrick Lamar)
1 	 Malcolm X
1 	 The Mantra
1 	 King's Dead (with Kendrick Lamar, Future & James Blake)
1 	 All The Stars (with SZA)
1 	 HUMBLE. (SKRILLEX REMIX)
1 	 The Heart Part 4
1 	 untitled 07 | levitate
1 	 These Walls
1 	 Alright
1 	 Swimming Pools (Drank) [Black Hippy Remix]
1 	 County Building Blues
1 	 i
1 	 Two Presidents
1 	 Swimming Pools (Drank)
1 	 My People (feat. Jay Rock) ("Bastards of the Party" Original Soundtrack Version)
1 	 She Needs Me [Remix] (Ringtone)
18 	 Starboy
30 	 MUSIC
14 	 Ctrl
17 	 WE DON'T TRUST YOU
11 	 Ca$i

### Extraindo IDs das Faixas

In [9]:
track_ids = []
for tracks in all_tracks:
    album_tracks = []
    for track in tracks.get('items'):
        album_tracks.append(track.get('uri'))
    track_ids.append(album_tracks)

In [10]:
track_objects = []
for track_id_list in track_ids:
    tracks = sp.tracks(track_id_list)
    track_objects.append(tracks)

### Buscando Detalhes das Faixas

In [11]:
from spotipy.exceptions import SpotifyException

audio_feature_objects = []
audio_features_available = True

for track_id_list in track_ids:
    try:
        features = sp.audio_features(track_id_list)
        audio_feature_objects.append(features)
    except SpotifyException as e:
        if e.http_status == 403:
            print("⚠️ Audio features endpoint deprecated (403). Skipping audio features.")
            audio_features_available = False
            break
        raise

if audio_features_available:
    print(f"Got audio features for {len(audio_feature_objects)} albums")
else:
    print("Continuing without audio features (still have track info + lyrics)")

HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=5gOfC9UzZQzTyShqPMrpjT,0nj9Bq5sHDiTxSHunhgkFb,45J4avUb9Ni0bnETYaYFVJ,5ho7VSXSmI2KM2nDjcnLyz,5S8VwnB4sLi6W0lYTWYylu,0RgjEkSbeuStKfT2Pa4Zai,0aB0v4027ukVziUGwVGYpG,4K1Pg0FLno1ltzX3jeqT83,2Uts1QFB4u2YNIMiqcb4de,1SGvjfc85yzqKXsfKcCxn2,3aZptNYC6Z1YoumeqZcDcQ,0wgOhYnqZKjOHr6bmdz0aN with Params: {} returned 403 due to None


⚠️ Audio features endpoint deprecated (403). Skipping audio features.
Continuing without audio features (still have track info + lyrics)


In [12]:
spotify_data = {
    "tracks": track_objects,
    "audio_features": audio_feature_objects if audio_features_available else None
}

with open(OUTPUT_DIR / "spotify.json", "w") as outfile:
    json.dump(spotify_data, outfile)
print(f"Saved to {OUTPUT_DIR / 'spotify.json'}")

Saved to output/spotify.json


### Buscando Features de Áudio

In [13]:
rows = []

if audio_features_available and spotify_data.get('audio_features'):
    for album_info, album_features in zip(
            spotify_data.get('tracks'), 
            spotify_data.get('audio_features')
            ):
        for track_info, track_features in zip(
            album_info.get('tracks'),
            album_features
            ):
            if track_features is None:
                continue
            rows.append({
                'name': track_info['name'],
                'duration_ms': track_info['duration_ms'],
                'popularity': track_info['popularity'],
                'num_markets': len(track_info['available_markets']),
                'album': track_info['album']['name'],
                'disc_number': track_info['disc_number'],
                'is_explicit': track_info['explicit'],
                'track_number': track_info['track_number'],
                'release_date': track_info['album']['release_date'],
                'artist': track_info['artists'][0]['name'],
                'danceability': track_features['danceability'],
                'energy': track_features['energy'],
                'key': track_features['key'],
                'loudness': track_features['loudness'],
                'mode': track_features['mode'],
                'speechiness': track_features['speechiness'],
                'acousticness': track_features['acousticness'],
                'instrumentalness': track_features['instrumentalness'],
                'liveness': track_features['liveness'],
                'valence': track_features['valence'],
                'tempo': track_features['tempo'],
                'time_signature': track_features['time_signature'],
            })
else:
    for album_info in spotify_data.get('tracks'):
        for track_info in album_info.get('tracks'):
            rows.append({
                'name': track_info['name'],
                'duration_ms': track_info['duration_ms'],
                'popularity': track_info['popularity'],
                'num_markets': len(track_info['available_markets']),
                'album': track_info['album']['name'],
                'disc_number': track_info['disc_number'],
                'is_explicit': track_info['explicit'],
                'track_number': track_info['track_number'],
                'release_date': track_info['album']['release_date'],
                'artist': track_info['artists'][0]['name'],
            })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_DIR / "spotify.csv", index=False)
print(f"Saved {len(df)} tracks to {OUTPUT_DIR / 'spotify.csv'}")
print(f"Columns: {list(df.columns)}")
df.head()

Saved 466 tracks to output/spotify.csv
Columns: ['name', 'duration_ms', 'popularity', 'num_markets', 'album', 'disc_number', 'is_explicit', 'track_number', 'release_date', 'artist']


,name,duration_ms,popularity,num_markets,album,disc_number,is_explicit,track_number,release_date,artist
0,wacced out murals,317092,69,183,GNX,1,True,1,2024-11-22,Kendrick Lamar
1,squabble up,157992,80,183,GNX,1,True,2,2024-11-22,Kendrick Lamar
2,luther (with sza),177598,83,183,GNX,1,False,3,2024-11-22,Kendrick Lamar
3,man at the garden,233173,66,183,GNX,1,True,4,2024-11-22,Kendrick Lamar
4,hey now (feat. dody6),217478,71,183,GNX,1,True,5,2024-11-22,Kendrick Lamar


## Pegando letras das músicas pelo Genius

### Salvando Dados do Spotify

In [14]:
api = genius.Genius(GENIUS_TOKEN, timeout=30, retries=3)
api.verbose = False
api.remove_section_headers = False  # Keep [Verse], [Chorus], etc.

# Load Spotify tracks to search for
spotify_df = pd.read_csv(OUTPUT_DIR / "spotify.csv")
unique_tracks = spotify_df.drop_duplicates(subset=['name'])[['name', 'album', 'artist']].to_dict('records')
print(f"Found {len(unique_tracks)} unique tracks from Spotify")

# Filter to main artist only
main_artist_tracks = [t for t in unique_tracks if ARTIST_NAME.lower() in t['artist'].lower()]
print(f"Tracks by {ARTIST_NAME}: {len(main_artist_tracks)}")

Found 428 unique tracks from Spotify
Tracks by Kendrick Lamar: 127


### Buscando Cada Faixa no Genius

In [15]:
# Search Genius for each Spotify track
from tqdm import tqdm

lyrics_data = []
missing_tracks = []

for track in tqdm(main_artist_tracks, desc="Fetching lyrics from Genius"):
    track_name = track['name']
    try:
        song = api.search_song(track_name, ARTIST_NAME)
        if song and song.lyrics:
            lyrics_data.append({
                'music_title': track_name,
                'genius_title': song.title,
                'lyrics': song.lyrics,
                'year': getattr(song, 'year', None),
                'album': track['album']
            })
        else:
            missing_tracks.append(track_name)
        time.sleep(0.3)  # Rate limit - 3 requests per second
    except Exception as e:
        print(f"\nFailed: {track_name} - {e}")
        missing_tracks.append(track_name)
        continue

print(f"\nFound lyrics for {len(lyrics_data)}/{len(main_artist_tracks)} tracks")
print(f"Missing: {len(missing_tracks)} tracks")

Fetching lyrics from Genius: 100%|██████████| 127/127 [08:06<00:00,  3.83s/it]


Found lyrics for 126/127 tracks
Missing: 1 tracks


### Salvando Letras em CSV

In [16]:
# Save lyrics to CSV (direct from search results)
lyric_df = pd.DataFrame(lyrics_data)
lyric_df.to_csv(OUTPUT_DIR / "lyrics.csv", index=False, encoding='utf-8-sig')
print(f"Saved {len(lyric_df)} songs to {OUTPUT_DIR / 'lyrics.csv'}")

# Save missing tracks
if missing_tracks:
    missing_df = pd.DataFrame({'track_name': missing_tracks})
    missing_df.to_csv(OUTPUT_DIR / "lyrics_missing.csv", index=False)
    print(f"Saved {len(missing_tracks)} missing tracks to {OUTPUT_DIR / 'lyrics_missing.csv'}")

lyric_df.head()

Saved 126 songs to output/lyrics.csv
Saved 1 missing tracks to output/lyrics_missing.csv


,music_title,genius_title,lyrics,year,album
0,wacced out murals,wacced out murals,[Intro: Deyra Barrera]\nSiento aquí tu presenc...,None,GNX
1,squabble up,squabble up,[Intro: Kendrick Lamar]\nGod knows\nI am reinc...,None,GNX
2,luther (with sza),KENDRICK LAMAR & SZA luther (REMIX) FEATURING ...,[Intro]\nIf this world were mine\n\n[Verse 1: ...,None,GNX
3,man at the garden,man at the garden,[Verse 1]\nTwice emotional stability\nOf sound...,None,GNX
4,hey now (feat. dody6),Kendrick Lamar - hey now ft. Dody6 (Traducción...,"[Letra de ""Kendrick Lamar - hey now""]\n\n[Intr...",None,GNX


### Montando CSV do Spotify